# ThinkPRM — Week 1-2 kickoff (Kaggle)

Setup: **Accelerator = GPU T4 x2**, **Internet = ON**.
Attach the `thinkprm-repo` dataset (repo zip), then run cells top to bottom.
Budget: ~15-20 of the 30 weekly GPU hours.

In [ ]:
# 0. Unpack repo + install (slim requirements, no vLLM)
!unzip -q -o /kaggle/input/thinkprm-repo/repo.zip -d /kaggle/working/
%cd /kaggle/working/thinkprm
!pip install -q -r requirements-kaggle.txt
!python -c "import torch; print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0))"

In [ ]:
# 1. Data pipeline (~15 min)
!python scripts/data/prepare_prm800k.py --out data/prm800k_unified.jsonl
!python scripts/data/prepare_mathshepherd.py --out data/mathshepherd_unified.jsonl --max_records 20000

✅ **Checkpoint 1**: PRM800K `gold_0` ≈ 40% of steps; Math-Shepherd `parse_failed` < 1%.

In [ ]:
# 2. Parser sanity check (CPU, instant)
import sys; sys.path.insert(0, "scripts/train")
from rewards import build_verifier_prompt, parse_verification
demo = "Step 1: correct\nStep 2: incorrect\nFirst error step: 2\nFinal: incorrect\nConfidence: 0.82"
print(parse_verification(demo))
assert parse_verification("garbage") is None
print("parser OK")

In [ ]:
# 3. DiscPRM baseline (~4-6 GPU hours)
!python scripts/train/train_discprm.py --config configs/discprm_1p5b.yaml

✅ **Checkpoint 2**: loss decreases, no NaN.

In [ ]:
# 4. SFT baseline = rejection-sampling proxy (~4-6 GPU hours)
!python scripts/train/train_sft.py --config configs/sft_1p5b.yaml

In [ ]:
# 5. Evaluate SFT (transformers, no vLLM; ~30-60 min for 200 items)
!python scripts/evaluation/evaluate_hf.py --model outputs/sft-1p5b --data data/prm800k_unified.jsonl --split phase1_test --max_n 200

✅ **Checkpoint 3**: record **step accuracy / solution accuracy / ECE / Brier** — the baseline Part A must beat.

Optional if quota remains: GRPO smoke test (~1 GPU hour, validates the reward loop):
`!python scripts/train/train_grpo.py --config configs/grpo_pilot.yaml` — watch r_agree/r_cal rise for ~50 steps, then stop.